# 01 - data exploration
carrefour data challenge — initial data loading and exploration.

## one-time setup: convert raw csv to parquet
run this cell once after placing the raw csv files in `data/raw/csv/`. safe to re-run — already-converted files are skipped.

In [ ]:
import sys
sys.path.append('..')

from src.data_loader import convert_csv_to_parquet

convert_csv_to_parquet()

In [ ]:
# file metadata — rows and columns without loading data
import pyarrow.parquet as pq
from pathlib import Path

_parquet_dir = Path("../data/raw/parquet")

pf = pq.ParquetFile(_parquet_dir / "linea_tickets.parquet")
print("linea_tickets:", pf.metadata.num_rows, "rows ×", pf.metadata.num_columns, "cols")

pf2 = pq.ParquetFile(_parquet_dir / "maestra_articulos.parquet")
print("maestra_articulos:", pf2.metadata.num_rows, "rows ×", pf2.metadata.num_columns, "cols")

**Result:**
- `linea_tickets`: 110,375,977 rows × 10 cols
- `maestra_articulos`: 893,944 rows × 4 cols

## load data

In [ ]:
# load product master (pandas, 34 MB) and tickets as lazy frame (polars)
import polars as pl
from src.data_loader import load_maestra_articulos

df_articulos = load_maestra_articulos()  # pandas — 34 MB, safe to load fully

df_tickets = pl.scan_parquet("../data/raw/parquet/linea_tickets.parquet")

schema = df_tickets.collect_schema()
print("articulos:", df_articulos.shape)
print("tickets columns:", schema.names())  # resolves schema only, no rows loaded

**Result:**
- `df_articulos` shape: (893944, 4)
- `df_tickets` columns: `idempres`, `fecha`, `hora`, `ticket`, `cliente`, `idarticu`, `unidades`, `importe`, `idpromoc`, `idtiprod`

## maestra_articulos
product catalogue — 893k articles across sectors.

In [ ]:
# schema and null check
df_articulos.dtypes.to_frame("dtype").assign(nulls=df_articulos.isnull().sum())

**maestra_articulos schema — no nulls:**

| column | dtype | nulls |
|---|---|---|
| idarticu | int64 | 0 |
| desc_larga_articulo | str | 0 |
| idsector | int64 | 0 |
| desc_sector | str | 0 |

In [ ]:
# products per sector
df_articulos.groupby(["idsector", "desc_sector"]).size().rename("n_products").sort_values(ascending=False)

**Products per sector:**

| idsector | desc_sector | n_products |
|---|---|---|
| 3 | BAZAR | 425,050 |
| 1 | P.G.C. | 171,672 |
| 6 | TEXTIL | 132,735 |
| 2 | PROD. FRESCOS TRADIC | 88,033 |
| 4 | ELECTROFOTO | 75,979 |
| 7 | GASOLINERA | 475 |

In [ ]:
df_articulos.head()

**maestra_articulos — first 5 rows:**

| idarticu | desc_larga_articulo | idsector | desc_sector |
|---|---|---|---|
| 966557 | GUISO DE VERDURAS LITORAL 415 GR | 1 | P.G.C. |
| 909163 | LA HISTORIA DE LOS 3 CERDITOS ANAYA | 3 | BAZAR |
| 594029 | MUU (SUSAETA) | 3 | BAZAR |
| 86412 | ALAS DE MOSCA PARA ÁNGEL ANAYA EDUCACIÓN | 3 | BAZAR |
| 507503 | CAMISETA NINO MANGA LARGA SPIDERMAN TINTA METALICA | 6 | TEXTIL |

## linea_tickets
110m transaction lines — the core input for the pipeline.

In [ ]:
# first 5 rows of tickets (lazy head, no full scan)
df_tickets.head(5).collect()

**linea_tickets — first 5 rows:**

| idempres | fecha | hora | ticket | cliente | idarticu | unidades | importe | idpromoc | idtiprod |
|---|---|---|---|---|---|---|---|---|---|
| 7 | 2022-01-27 | 1921 | 2022-01-270025002502... | f228d040... | 819748 | 2 | 3.64 | Promo | 2 |
| 7 | 2022-06-20 | 2046 | 2022-06-200050005003... | b0b59dac... | 507244 | 1 | 2.10 | No promo | 1 |
| 7 | 2022-05-19 | 1824 | 2022-05-190099009900... | 7a253da3... | 649683 | 1 | 1.45 | No promo | 1 |
| 7 | 2022-03-26 | 1146 | 2022-03-260070007001... | f9819591... | 65358 | 1 | 2.61 | Promo | 2 |
| 2 | 2022-02-13 | 1851 | 2022-02-136818681815... | 37b896c5... | 261966 | 1 | 1.01 | No promo | 1 |

In [ ]:
# schema and null check — streaming, never loads full dataset into memory
null_counts = (
    df_tickets
    .select(pl.all().is_null().sum())
    .collect(engine="streaming")
)

schema = df_tickets.collect_schema()
pl.DataFrame({
    "column": schema.names(),
    "dtype": [str(schema[c]) for c in schema.names()],
    "nulls": null_counts.row(0),
})

**linea_tickets schema — no nulls:**

| column | dtype | nulls |
|---|---|---|
| idempres | Int64 | 0 |
| fecha | Date | 0 |
| hora | Int64 | 0 |
| ticket | String | 0 |
| cliente | String | 0 |
| idarticu | Int64 | 0 |
| unidades | Int64 | 0 |
| importe | Float64 | 0 |
| idpromoc | String | 0 |
| idtiprod | Int64 | 0 |

In [ ]:
# dataset scale
stats = df_tickets.select([
    pl.col("cliente").n_unique().alias("unique_customers"),
    pl.col("idarticu").n_unique().alias("unique_products"),
    pl.col("ticket").n_unique().alias("unique_tickets"),
    pl.col("fecha").min().alias("fecha_min"),
    pl.col("fecha").max().alias("fecha_max"),
    pl.col("idempres").n_unique().alias("unique_stores"),
]).collect(engine="streaming")

print(f"unique customers : {stats['unique_customers'][0]:>12,}")
print(f"unique products  : {stats['unique_products'][0]:>12,}")
print(f"unique tickets   : {stats['unique_tickets'][0]:>12,}")
print(f"date range       :  {stats['fecha_min'][0]} → {stats['fecha_max'][0]}")
print(f"stores (idempres): {stats['unique_stores'][0]:>12,}")

**Dataset scale:**

| metric | value |
|---|---|
| unique customers | 1,460,657 |
| unique products | 111,802 |
| unique tickets | 18,501,515 |
| date range | 2022-01-01 → 2022-06-30 |
| stores (idempres) | 4 |

In [ ]:
# promo vs non-promo split — lazy group-by, no full collect
(
    df_tickets
    .select("idpromoc")
    .group_by("idpromoc")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("share (%)")
    )
    .drop("count")
    .sort("share (%)", descending=True)
    .collect(engine="streaming")
)

**Promo vs non-promo split:**

| idpromoc | share (%) |
|---|---|
| No promo | 77.6 |
| Promo | 22.4 |

In [20]:
df_tickets.shape

(191017715, 10)

## pre-merge quality checks
before joining, verify join key integrity and flag data issues that affect the pipeline.

In [ ]:
# 1. join key must be unique in the product master
assert df_articulos["idarticu"].is_unique, "duplicate idarticu in product master!"
print("articulos join key: unique ✓")

# 2. tickets products not in the master (would produce nulls after left join)
orphaned = set(df_tickets["idarticu"].unique()) - set(df_articulos["idarticu"])
print(f"orphaned idarticu (in tickets, not in master): {len(orphaned):,}")

# 3. returns / cancellations — negative unidades or importe
print(f"negative unidades : {(df_tickets['unidades'] < 0).sum():,}")
print(f"negative importe  : {(df_tickets['importe'] < 0).sum():,}")

# 4. fecha dtype — needs to be datetime for temporal features
print(f"fecha dtype       : {df_tickets['fecha'].dtype}")

## merge + clean
left join on `idarticu` — enrich ticket lines with product name, sector, and sector description.
`fecha` is formatted as dd.mm.yyyy, `hora` as hh:mm.

In [ ]:
import pandas as pd

# format fecha: "2022-01-27" → "27.01.2022"
df_tickets["fecha"] = pd.to_datetime(df_tickets["fecha"]).dt.strftime("%d.%m.%Y")

# format hora: 1921 → "19:21"
df_tickets["hora"] = (
    df_tickets["hora"].astype(str).str.zfill(4).apply(lambda x: x[:2] + ":" + x[2:])
)

# left join — include product name so each line shows what was bought
df = df_tickets.merge(
    df_articulos[["idarticu", "desc_larga_articulo", "idsector", "desc_sector"]],
    on="idarticu",
    how="left",
    validate="m:1",
)

assert df["idsector"].isna().sum() == 0, "orphaned products present — some ticket lines have no matching article"

print("merge complete")
print(f"shape : {df.shape}")
print(f"columns: {df.columns.tolist()}")

In [ ]:
# schema and null check on the merged table
df.dtypes.to_frame("dtype").assign(nulls=df.isnull().sum())

In [ ]:
# who / when / what — one row per transaction line
df[[
    "cliente",         # who
    "fecha", "hora",   # when
    "ticket", "idempres",  # transaction id + store
    "idarticu", "desc_larga_articulo", "desc_sector",  # what
    "unidades", "importe", "idpromoc",  # how much / promo flag
]].head(10)